In [ ]:
import os

if not os.path.exists('SMSSpamCollection'):
    !wget -q "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"
    !unzip -q smsspamcollection.zip
    print("✅ Dataset downloaded successfully!")
else:
    print("✅ Dataset already exists, skipping download!")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("✅ All libraries imported!")

In [ ]:
# Load the dataset
df = pd.read_csv('SMSSpamCollection', sep='\t', header=None, names=['label', 'message'])

print("Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Check how many spam vs ham
print(df['label'].value_counts())

# Visualize
df['label'].value_counts().plot(kind='bar', color=['steelblue','tomato'])
plt.title('Ham vs Spam Count')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Convert labels to numbers: ham=0, spam=1
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

# Add message length as a feature (optional but insightful)
df['msg_length'] = df['message'].apply(len)

print("✅ Preprocessing done!")
df.head()

In [ ]:
# Split data
X = df['message']
y = df['label_num']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Convert text to numbers using TF-IDF
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train_vec, y_train)
nb_preds = nb_model.predict(X_test_vec)

# Train SVM
svm_model = LinearSVC()
svm_model.fit(X_train_vec, y_train)
svm_preds = svm_model.predict(X_test_vec)

print("✅ Models trained!")

In [ ]:
print("=" * 40)
print("NAIVE BAYES RESULTS")
print("=" * 40)
print(f"Accuracy: {accuracy_score(y_test, nb_preds)*100:.2f}%")
print(classification_report(y_test, nb_preds, target_names=['Ham','Spam']))

print("=" * 40)
print("SVM RESULTS")
print("=" * 40)
print(f"Accuracy: {accuracy_score(y_test, svm_preds)*100:.2f}%")
print(classification_report(y_test, svm_preds, target_names=['Ham','Spam']))

In [ ]:
def predict_spam(message):
    msg_vec = vectorizer.transform([message])
    nb_result = nb_model.predict(msg_vec)[0]
    svm_result = svm_model.predict(msg_vec)[0]
    print(f"Message: '{message}'")
    print(f"  Naive Bayes → {'🚨 SPAM' if nb_result == 1 else '✅ HAM (Not Spam)'}")
    print(f"  SVM         → {'🚨 SPAM' if svm_result == 1 else '✅ HAM (Not Spam)'}")
    print()

# Test it!
predict_spam("Congratulations! You've won a FREE iPhone. Click here now!")
predict_spam("Hey, are we still meeting for lunch tomorrow?")
predict_spam("URGENT: Your bank account has been compromised. Call now!")